<a href="https://colab.research.google.com/github/tthuy123/graduation/blob/main/MODEL_TRAINING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

So this version has the best accuracy until now that i discovered on WF-PD classification task on OULAD with result reach 0.798 accuracy at 20% (8 weeks) data early prediction



In [1]:
import os
import random
import numpy as np
import pandas as pd

from dataclasses import dataclass
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import Dense, Dropout, LSTM, Conv1D, MaxPooling1D, Concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


In [2]:
# =====================================================
# CONFIG
# =====================================================

@dataclass
class Config:
    data_dir: str = "/content/drive/MyDrive/OULAD"
    max_weeks: int = 40
    test_size: float = 0.2
    random_state: int = 42
    learning_rate: float = 1e-3
    batch_size: int = 64
    epochs: int = 25


CFG = Config()

EARLY_MARKERS = [0.1,0.2, 0.4, 0.6, 0.8, 1.0]

In [3]:
# =====================================================
# SEED
# =====================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

In [4]:
# =====================================================
# UTIL
# =====================================================

def map_day_to_week(day, max_weeks):
    if pd.isna(day) or day < 0:
        return 0
    week = int(day // 7)
    return min(week, max_weeks - 1)

In [5]:
# =====================================================
# LOAD DATA
# =====================================================

def load_data(data_dir):

    files = {
        "student_info": "studentInfo.csv",
        "student_vle": "studentVle.csv",
        "student_assessment": "studentAssessment.csv",
        "assessment": "assessments.csv",
        "vle": "vle.csv"
    }

    data = {}

    for k, v in files.items():
        path = os.path.join(data_dir, v)
        if not os.path.exists(path):
            raise FileNotFoundError(path)
        data[k] = pd.read_csv(path)

    return data

In [6]:
load_data(
    data_dir=CFG.data_dir
)

{'student_info':       code_module code_presentation  id_student gender                region  \
 0             AAA             2013J       11391      M   East Anglian Region   
 1             AAA             2013J       28400      F              Scotland   
 2             AAA             2013J       30268      F  North Western Region   
 3             AAA             2013J       31604      F     South East Region   
 4             AAA             2013J       32885      F  West Midlands Region   
 ...           ...               ...         ...    ...                   ...   
 32588         GGG             2014J     2640965      F                 Wales   
 32589         GGG             2014J     2645731      F   East Anglian Region   
 32590         GGG             2014J     2648187      F          South Region   
 32591         GGG             2014J     2679821      F     South East Region   
 32592         GGG             2014J     2684003      F      Yorkshire Region   
 
          

In [7]:
# =====================================================
# TARGET
# =====================================================

def build_target(df):

    df = df.copy()

    df["target"] = df["final_result"].apply(
        lambda x: 1 if x in ["Pass", "Distinction"] else 0
    )

    return df

In [8]:
# =====================================================
# CLICK SERIES
# =====================================================

def build_click_series(df_student_vle, max_weeks):

    df = df_student_vle.copy()

    df["week_idx"] = df["date"].apply(
        lambda x: map_day_to_week(x, max_weeks)
    )

    grouped = df.groupby(
        ["id_student", "code_module", "code_presentation", "week_idx"]
    )["sum_click"].sum().reset_index()

    pivot = grouped.pivot_table(
        index=["id_student", "code_module", "code_presentation"],
        columns="week_idx",
        values="sum_click",
        fill_value=0
    )

    pivot = pivot.reindex(columns=list(range(max_weeks)), fill_value=0)

    pivot.columns = [f"click_wk_{i}" for i in pivot.columns]

    return pivot.reset_index()

In [9]:
# =====================================================
# ACTIVITY FEATURES
# =====================================================

def build_activity_features(student_vle, vle, max_weeks):

    # merge activity type
    df = student_vle.merge(
        vle[["id_site", "activity_type"]],
        on="id_site",
        how="left"
    )

    # map day -> week
    df["week_idx"] = df["date"].apply(
        lambda x: map_day_to_week(x, max_weeks)
    )

    # aggregate clicks per activity type per week
    activity = df.pivot_table(
        index=["id_student","code_module","code_presentation","week_idx"],
        columns="activity_type",
        values="sum_click",
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    # =====================================================
    # get all activity types dynamically
    # =====================================================

    activity_types = vle["activity_type"].unique().tolist()

    # ensure all activity columns exist
    for col in activity_types:
        if col not in activity.columns:
            activity[col] = 0

    # =====================================================
    # TOTAL CLICKS
    # =====================================================

    activity["total"] = activity[activity_types].sum(axis=1)

    # inactive feature
    activity["inactive"] = (activity["total"] == 0).astype(int)

    # avoid divide-by-zero
    activity["total"] = activity["total"].replace(0,1)

    # =====================================================
    # RATIOS
    # =====================================================

    for col in activity_types:
        activity[f"{col}_ratio"] = activity[col] / activity["total"]

    ratio_cols = [f"{col}_ratio" for col in activity_types]

    # =====================================================
    # ENTROPY
    # =====================================================

    def entropy(row):

        vals = row[activity_types].values.astype(float)

        total = vals.sum()

        if total == 0:
            return np.nan

        p = vals / total
        p = p[p > 0]

        return -(p * np.log(p)).sum()

    activity["entropy"] = activity.apply(entropy, axis=1)

    # =====================================================
    # FEATURES USED IN MODEL
    # =====================================================

    feature_cols = ratio_cols + ["entropy", "inactive"]

    # =====================================================
    # WEEKLY PIVOT
    # =====================================================

    pivot = activity.pivot_table(
        index=["id_student","code_module","code_presentation"],
        columns="week_idx",
        values=feature_cols,
        fill_value=0
    )

    # =====================================================
    # ENSURE FULL WEEK RANGE
    # =====================================================

    full_cols = pd.MultiIndex.from_product(
        [
            feature_cols,
            range(max_weeks)
        ]
    )

    pivot = pivot.reindex(columns=full_cols, fill_value=0)

    # =====================================================
    # FLATTEN COLUMN NAMES
    # =====================================================

    pivot.columns = [
        f"{feat}_wk_{week}"
        for feat, week in pivot.columns
    ]

    # fill entropy NaN
    entropy_cols = [c for c in pivot.columns if "entropy" in c]
    pivot[entropy_cols] = pivot[entropy_cols].fillna(0)

    return pivot.reset_index()

In [10]:
def build_assessment_series_safe(df_student_assessment, df_assessment, max_weeks, cutoff_week):
    """
    Args:
        df_student_assessment: Bảng kết quả nộp bài của sinh viên.
        df_assessment: Bảng thông tin các bài kiểm tra (có cột 'date' là hạn chót).
        max_weeks: Tổng số tuần tối đa của khóa học.
        cutoff_week: Tuần hiện tại (chỉ lấy dữ liệu trước tuần này để dự báo).
    """

    # 1. Kết hợp dữ liệu
    merged = df_student_assessment.merge(df_assessment, on="id_assessment")

    # 2. XÁC ĐỊNH TUẦN DỰA TRÊN HẠN CHÓT
    # Giả sử 1 tuần có 7 ngày. Bài kiểm tra có 'date' rơi vào ngày nào thì thuộc tuần đó.
    merged["date"] = merged["date"].fillna(0)
    merged["week_idx"] = (merged["date"] // 7).astype(int)

    # 3. CHẶN DỮ LIỆU TƯƠNG LAI
    # Chỉ giữ lại các bài kiểm tra có hạn chót nằm trong khoảng từ tuần 0 đến cutoff_week
    merged = merged[merged["week_idx"] < cutoff_week].copy()

    # 4. TÍNH TOÁN LATENESS AN TOÀN
    # Nếu chưa nộp (score NaN), ta coi là trễ so với mốc cutoff hoặc ngày hạn chót
    def calc_lateness(row):
        if pd.isna(row["date_submitted"]):
            # Nếu chưa nộp bài đã quá hạn, tính độ trễ tới thời điểm hiện tại (cutoff)
            cutoff_day = cutoff_week * 7
            return max(0, cutoff_day - row["date"])
        else:
            # Nếu đã nộp, tính độ trễ thực tế (có thể âm nếu nộp sớm)
            return row["date_submitted"] - row["date"]

    merged["lateness"] = merged.apply(calc_lateness, axis=1)
    merged["score"] = merged["score"].fillna(0)
    merged["weight"] = merged["weight"].fillna(0)

    # 5. TẠO BẢNG XOAY (Pivot Tables)
    idx = ["id_student", "code_module", "code_presentation"]
    # Chỉ tạo các cột từ tuần 0 đến cutoff_week - 1
    target_weeks = list(range(cutoff_week))

    # Điểm số trung bình theo tuần
    score_pivot = merged.pivot_table(
        index=idx, columns="week_idx", values="score", aggfunc="mean"
    ).reindex(columns=target_weeks, fill_value=0)

    # Độ trễ trung bình theo tuần
    late_pivot = merged.pivot_table(
        index=idx, columns="week_idx", values="lateness", aggfunc="mean"
    ).reindex(columns=target_weeks, fill_value=0)

    # Trọng số bài thi trong tuần
    weight_pivot = merged.pivot_table(
        index=idx, columns="week_idx", values="weight", aggfunc="max"
    ).reindex(columns=target_weeks, fill_value=0)

    # 6. ĐỔI TÊN CỘT ĐỂ PHÂN BIỆT
    score_pivot.columns = [f"score_wk_{i}" for i in score_pivot.columns]
    late_pivot.columns = [f"late_wk_{i}" for i in late_pivot.columns]
    weight_pivot.columns = [f"weight_wk_{i}" for i in weight_pivot.columns]

    # 7. GỘP LẠI THÀNH DATAFRAME CUỐI CÙNG
    df_final = score_pivot.join(late_pivot).join(weight_pivot).reset_index()

    return df_final

In [11]:
# =====================================================
# BUILD DATASET
# =====================================================

def build_dataset(data, cutoff_week):
    df_info = build_target(data["student_info"])

    df_click = build_click_series(data["student_vle"], CFG.max_weeks)

    df_assess = build_assessment_series_safe(
        data["student_assessment"],
        data["assessment"],
        CFG.max_weeks,
        cutoff_week
    )

    df_activity = build_activity_features(
        data["student_vle"],
        data["vle"],
        CFG.max_weeks
    )

    keys = ["id_student", "code_module", "code_presentation"]

    df = df_info.merge(df_click, on=keys, how="left")
    df = df.merge(df_assess, on=keys, how="left")
    df = df.merge(df_activity, on=keys, how="left")

    df = df.fillna(0)
    return df

In [12]:
# =====================================================
# PREPARE FEATURES
# =====================================================

def prepare_features(df, activity_types, cutoff_week):

    base_features = ["click", "score", "late", "weight", "entropy", "inactive"]
    ratio_features = [f"{act}_ratio" for act in activity_types]

    all_feature_types = base_features + ratio_features

    temporal_data = {}
    all_temporal_cols = []

    for feat in all_feature_types:

        arr = np.zeros((len(df), cutoff_week))

        for i in range(cutoff_week):
            col = f"{feat}_wk_{i}"

            if col in df.columns:
                arr[:, i] = df[col].values
                all_temporal_cols.append(col)

        temporal_data[feat] = arr

    exclude_from_static = [
        "id_student",
        "code_module",
        "code_presentation",
        "final_result",
        "target"
    ] + all_temporal_cols

    obj_cols = [
        c for c in df.select_dtypes(include=["object"]).columns
        if c not in exclude_from_static
    ]

    df_processed = pd.get_dummies(df, columns=obj_cols, drop_first=True)

    static_cols = [
        c for c in df_processed.columns
        if c not in exclude_from_static
    ]

    X_static = df_processed[static_cols].astype(np.float32).values
    y = df_processed["target"].astype(np.int32).values

    return X_static, temporal_data, y

In [13]:
# =====================================================
# TEMPORAL TENSOR
# =====================================================

def create_temporal_tensor(temporal_data):

    if len(temporal_data) == 0:
        raise ValueError("temporal_data is empty")

    first_key = list(temporal_data.keys())[0]

    n = temporal_data[first_key].shape[0]
    weeks = temporal_data[first_key].shape[1]

    tensors = []

    for key in sorted(temporal_data.keys()):
        arr = temporal_data[key]

        if arr.shape[1] != weeks:
            raise ValueError(f"Week mismatch in {key}")

        tensors.append(arr.reshape(n, weeks, 1))

    X_temporal = np.concatenate(tensors, axis=2)

    return X_temporal.astype(np.float32)

In [14]:
# =====================================================
# MODEL
# =====================================================

def build_model(temporal_shape, static_dim):
    # temporal_shape sẽ là (weeks, num_features)
    temporal_input = Input(shape=temporal_shape)

    # CNN layer để trích xuất đặc trưng không gian thời gian cục bộ
    x = Conv1D(64, 3, padding="same", activation="relu")(temporal_input)
    x = MaxPooling1D(2)(x)
    x = LSTM(128)(x)

    # Nhánh dữ liệu tĩnh (Static features)
    static_input = Input(shape=(static_dim,))
    s = Dense(64, activation="relu")(static_input)

    # Kết hợp hai nhánh
    merged = Concatenate()([x, s])

    z = Dense(64, activation="relu")(merged)
    z = Dropout(0.3)(z)

    output = Dense(1, activation="sigmoid")(z)

    model = Model(inputs=[temporal_input, static_input], outputs=output)

    model.compile(
        optimizer=Adam(learning_rate=CFG.learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [21]:
# =====================================================
# PURE LSTM MODEL
# =====================================================

def build_lstm_model(temporal_shape):

    temporal_input = Input(shape=temporal_shape)

    x = LSTM(128, return_sequences=True)(temporal_input)
    x = Dropout(0.3)(x)

    x = LSTM(64)(x)
    x = Dropout(0.3)(x)

    x = Dense(64, activation="relu")(x)

    output = Dense(1, activation="sigmoid")(x)

    model = Model(inputs=temporal_input, outputs=output)

    model.compile(
        optimizer=Adam(learning_rate=CFG.learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [22]:
# =====================================================
# CREATE ML DATASET (FLATTEN TEMPORAL)
# =====================================================

def create_ml_dataset(X_static, X_temporal):

    n = X_temporal.shape[0]

    # flatten temporal tensor
    X_temporal_flat = X_temporal.reshape(n, -1)

    # combine static + temporal
    X_ml = np.concatenate([X_static, X_temporal_flat], axis=1)

    return X_ml

# =====================================================
# ML MODELS
# =====================================================

def train_baselines(X_train, X_test, y_train, y_test):

    results = {}

    # -----------------------------
    # Logistic Regression
    # -----------------------------
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    lr = LogisticRegression(max_iter=500)

    lr.fit(X_train_scaled, y_train)

    y_pred = lr.predict(X_test_scaled)

    results["Logistic Regression"] = accuracy_score(y_test, y_pred)

    # -----------------------------
    # Random Forest
    # -----------------------------
    rf = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_test)

    results["Random Forest"] = accuracy_score(y_test, y_pred)

    # -----------------------------
    # XGBoost
    # -----------------------------
    xgb = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    )

    xgb.fit(X_train, y_train)

    y_pred = xgb.predict(X_test)

    results["XGBoost"] = accuracy_score(y_test, y_pred)

    return results

In [23]:
# =====================================================
# RUN BASELINE MODELS
# =====================================================

def run_baselines():

    set_seed()

    data = load_data(CFG.data_dir)

    activity_types = data["vle"]["activity_type"].unique().tolist()

    results_all = []

    for marker in EARLY_MARKERS:

        cutoff_week = int(CFG.max_weeks * marker)

        print("\n" + "="*40)
        print(f"BASELINE AT {int(marker*100)}% ({cutoff_week} weeks)")
        print("="*40)

        # -----------------------------
        # Build dataset
        # -----------------------------

        df = build_dataset(data, cutoff_week)

        X_static, temporal_data, y = prepare_features(
            df,
            activity_types,
            cutoff_week
        )

        X_temporal = create_temporal_tensor(temporal_data)

        # -----------------------------
        # Create ML dataset
        # -----------------------------

        X_temporal_flat = X_temporal.reshape(X_temporal.shape[0], -1)

        X_ml = np.concatenate([X_static, X_temporal_flat], axis=1)

        # -----------------------------
        # Train/test split
        # -----------------------------

        idx = np.arange(len(y))

        train_idx, test_idx = train_test_split(
            idx,
            test_size=CFG.test_size,
            random_state=CFG.random_state,
            stratify=y
        )

        X_train, X_test = X_ml[train_idx], X_ml[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        results = {}

        # =================================================
        # Logistic Regression
        # =================================================

        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        lr = LogisticRegression(max_iter=500)

        lr.fit(X_train_scaled, y_train)

        y_pred = lr.predict(X_test_scaled)

        acc = accuracy_score(y_test, y_pred)

        print(f"Logistic Regression: {acc:.4f}")
        print(classification_report(y_test, y_pred))

        results["Logistic Regression"] = acc

        # =================================================
        # Random Forest
        # =================================================

        rf = RandomForestClassifier(
            n_estimators=300,
            random_state=CFG.random_state,
            n_jobs=-1
        )

        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_test)

        acc = accuracy_score(y_test, y_pred)

        print(f"Random Forest: {acc:.4f}")
        print(classification_report(y_test, y_pred))

        results["Random Forest"] = acc

        # =================================================
        # XGBoost
        # =================================================

        xgb = XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=CFG.random_state
        )

        xgb.fit(X_train, y_train)

        y_pred = xgb.predict(X_test)

        acc = accuracy_score(y_test, y_pred)

        print(f"XGBoost: {acc:.4f}")
        print(classification_report(y_test, y_pred))

        results["XGBoost"] = acc

        results["marker"] = marker

        results_all.append(results)

    # =====================================================
    # Final comparison table
    # =====================================================

    print("\n" + "="*50)
    print("ML COMPARISON TABLE")
    print("="*50)

    df_results = pd.DataFrame(results_all)

    print(df_results)

    return df_results


In [24]:
# =====================================================
# RUN PURE LSTM MODEL
# =====================================================

def run_lstm():

    set_seed()

    # Load data
    data = load_data(CFG.data_dir)

    activity_types = data["vle"]["activity_type"].unique().tolist()

    for marker in EARLY_MARKERS:

        cutoff_week = int(CFG.max_weeks * marker)

        print("\n" + "="*40)
        print(f"LSTM EARLY PREDICTION AT {int(marker*100)}% ({cutoff_week} weeks)")
        print("="*40)

        # ------------------------------------------------
        # BUILD DATASET
        # ------------------------------------------------

        df = build_dataset(data, cutoff_week)

        X_static, temporal_data, y = prepare_features(
            df,
            activity_types,
            cutoff_week
        )

        X_temporal = create_temporal_tensor(temporal_data)

        num_temporal_features = X_temporal.shape[2]

        # ------------------------------------------------
        # TRAIN TEST SPLIT
        # ------------------------------------------------

        idx = np.arange(len(y))

        train_idx, test_idx = train_test_split(
            idx,
            test_size=CFG.test_size,
            random_state=CFG.random_state,
            stratify=y
        )

        X_temporal_train = X_temporal[train_idx]
        X_temporal_test = X_temporal[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        # ------------------------------------------------
        # CLASS WEIGHTS
        # ------------------------------------------------

        classes = np.unique(y_train)

        weights = compute_class_weight(
            class_weight="balanced",
            classes=classes,
            y=y_train
        )

        class_weights = {int(c): float(w) for c, w in zip(classes, weights)}

        # ------------------------------------------------
        # BUILD MODEL
        # ------------------------------------------------

        lstm_model = build_lstm_model(
            temporal_shape=(cutoff_week, num_temporal_features)
        )

        early_stop = EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        )

        # ------------------------------------------------
        # TRAIN
        # ------------------------------------------------

        lstm_model.fit(
            X_temporal_train,
            y_train,
            validation_data=(X_temporal_test, y_test),
            epochs=CFG.epochs,
            batch_size=CFG.batch_size,
            class_weight=class_weights,
            callbacks=[early_stop],
            verbose=1
        )

        # ------------------------------------------------
        # EVALUATE
        # ------------------------------------------------

        y_prob = lstm_model.predict(X_temporal_test, verbose=0)

        y_pred = (y_prob > 0.5).astype(int)

        print("\n[LSTM RESULT]")
        print(classification_report(y_test, y_pred))
        print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

In [19]:
def run():
    set_seed()

    # 1. Load dữ liệu gốc
    data = load_data(CFG.data_dir)

    # Lấy danh sách activity types để dùng cho toàn bộ pipeline
    activity_types = data["vle"]["activity_type"].unique().tolist()

    # 2. Vòng lặp qua các mốc dự đoán (Early Markers)
    for marker in EARLY_MARKERS:
        cutoff_week = int(CFG.max_weeks * marker)

        print("\n" + "="*40)
        print(f"EARLY PREDICTION AT {int(marker*100)}% ({cutoff_week} weeks)")
        print("="*40)

        # 3. Build Dataset: Quan trọng là dùng cutoff_week để chống leak bài kiểm tra
        # Mỗi mốc dự đoán sẽ có một bộ feature khác nhau
        df = build_dataset(data, cutoff_week=cutoff_week)

        # 4. Prepare Features: Tự động nhận diện các cột dựa trên activity_types
        X_static, temporal_data, y = prepare_features(
            df,
            activity_types,
            cutoff_week
        )
        # 5. Create Temporal Tensor: Chuyển dict thành 3D array (N, Weeks, Features)
        X_temporal = create_temporal_tensor(temporal_data)

        # Lấy số lượng feature thực tế sau khi đã xử lý dynamic
        num_temporal_features = X_temporal.shape[2]

        # 6. Train/Test Split
        idx = np.arange(len(y))
        train_idx, test_idx = train_test_split(
            idx,
            test_size=CFG.test_size,
            random_state=CFG.random_state,
            stratify=y
        )

        X_static_train, X_static_test = X_static[train_idx], X_static[test_idx]
        X_temporal_train, X_temporal_test = X_temporal[train_idx], X_temporal[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # 7. Scaling (Chỉ scale dựa trên tập Train)
        scaler = MinMaxScaler()
        X_static_train = scaler.fit_transform(X_static_train)
        X_static_test = scaler.transform(X_static_test)

        # 8. Class Weights (Xử lý dữ liệu lệch - Imbalanced data)
        classes = np.unique(y_train)
        weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
        class_weights = {int(c): float(w) for c, w in zip(classes, weights)}

        # 9. Build & Train Model
        # temporal_shape lúc này là (cutoff_week, num_features)
        model = build_model(
            temporal_shape=(cutoff_week, num_temporal_features),
            static_dim=X_static_train.shape[1]
        )

        early_stop = EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        )

        # Lưu ý: X_temporal lúc này đã có số tuần đúng bằng cutoff_week
        # do hàm build_assessment_series_safe và các hàm pivot đã reindex đúng số tuần
        model.fit(
            [X_temporal_train, X_static_train],
            y_train,
            validation_data=([X_temporal_test, X_static_test], y_test),
            epochs=CFG.epochs,
            batch_size=CFG.batch_size,
            class_weight=class_weights,
            callbacks=[early_stop],
            verbose=1
        )

        # 10. Đánh giá
        y_prob = model.predict([X_temporal_test, X_static_test], verbose=0)
        y_pred = (y_prob > 0.5).astype(int)

        print(f"\n[Kết quả mốc {int(marker*100)}%]")
        print(classification_report(y_test, y_pred))
        print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

In [20]:
if __name__ == "__main__":
    run()
    # run_baselines()
    # run_lstm()


EARLY PREDICTION AT 10% (4 weeks)
Epoch 1/25
408/408 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.8673 - loss: 0.3178 - val_accuracy: 0.9293 - val_loss: 0.1882
Epoch 2/25
408/408 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9298 - loss: 0.1883 - val_accuracy: 0.9310 - val_loss: 0.1853
Epoch 3/25
408/408 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9327 - loss: 0.1775 - val_accuracy: 0.9301 - val_loss: 0.1906
Epoch 4/25
408/408 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9350 - loss: 0.1668 - val_accuracy: 0.9316 - val_loss: 0.1930
Epoch 5/25
408/408 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9405 - loss: 0.1543 - val_accuracy: 0.9324 - val_loss: 0.1957
Epoch 6/25
408/408 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9487 - loss: 0.1370 - val_accuracy: 0.9325 - val_loss: 0.2002
Epoch 7/25
408/408 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9571 - loss: 0.1208 - val_accuracy: 0.9330 - val_loss: 0.2111

[Kết quả mốc 10%]
              precision    recall  f1-score   

In [25]:
if __name__ == "__main__":
    # run()
    run_baselines()
    run_lstm()


BASELINE AT 10% (4 weeks)
Logistic Regression: 0.9264
              precision    recall  f1-score   support

           0       0.93      0.93      0.93      3442
           1       0.92      0.93      0.92      3077

    accuracy                           0.93      6519
   macro avg       0.93      0.93      0.93      6519
weighted avg       0.93      0.93      0.93      6519

Random Forest: 0.9302
              precision    recall  f1-score   support

           0       0.99      0.88      0.93      3442
           1       0.88      0.99      0.93      3077

    accuracy                           0.93      6519
   macro avg       0.93      0.93      0.93      6519
weighted avg       0.94      0.93      0.93      6519

XGBoost: 0.9448
              precision    recall  f1-score   support

           0       0.98      0.92      0.95      3442
           1       0.91      0.98      0.94      3077

    accuracy                           0.94      6519
   macro avg       0.95      0.95  